**Set environment**

In [1]:
source ../run_config_project.sh
show_env

BASE DIRECTORY (FD_BASE):      /hpc/group/igvf/kk319
REPO DIRECTORY (FD_REPO):      /hpc/group/igvf/kk319/repo
WORK DIRECTORY (FD_WORK):      /hpc/group/igvf/kk319/work
DATA DIRECTORY (FD_DATA):      /hpc/group/igvf/kk319/data
CONTAINER DIR. (FD_SING):      /hpc/group/igvf/kk319/container

You are working with           
PATH OF PROJECT (FD_PRJ):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR
PROJECT RESULTS (FD_RES):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results
PROJECT SCRIPTS (FD_EXE):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts
PROJECT DATA    (FD_DAT):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/data
PROJECT NOTE    (FD_NBK):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/notebooks
PROJECT DOCS    (FD_DOC):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/docs
PROJECT LOG     (FD_LOG):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/log
PROJECT REF     (FD_REF):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/references
PR

## Preview

In [2]:
ls -1 ${FD_RES}

analysis_variant_motif_richard
analysis_variant_motif_richard_arc251231
predict_variant_alphagenome
predict_variant_kircher2019


In [3]:
ls ${FD_RES}/analysis_variant_motif_richard

background_zero_order.npy
background_zero_order.tsv
batches_dev
batches_pilot
batches_top
batches_top_dinuc
motifdelta_pilot_jvierstra_v2.1beta
motifdelta_top_jaspar2024
motifdelta_top_jvierstra_v2.1beta
motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl
motif_jaspar2024_core_vertebrates_nonredundant.pmap.pkl
motif_jaspar2024_core_vertebrates_nonredundant.tbind.pkl
motifmodel_pilot_jvierstra_v2.1beta
motif_nonredundant_jvierstra_v2.1beta.lods.pkl
motif_nonredundant_jvierstra_v2.1beta.pmap.pkl
motif_nonredundant_jvierstra_v2.1beta.tbind.pkl
motifscan_pilot_jvierstra_v2.1beta
motifscan_top_jaspar2024
motifscan_top_jvierstra_v2.1beta
variant_closed_gof_bluestarr.flankL35R70.ref.fa
variant_closed_gof_bluestarr.tsv


In [4]:
ls ${FD_RES}/analysis_variant_motif_richard/batches_top_dinuc

variant_closed_gof_bluestarr.flankL35R70.top01k.dinuc.ref.fa.gz
variant_closed_gof_bluestarr.flankL35R70.top10k.dinuc.ref.fa.gz


## Jaspar2024

### Prepare

In [5]:
FD_BATCH=${FD_RES}/analysis_variant_motif_richard/batches_top_dinuc
FD_MSCAN=${FD_RES}/analysis_variant_motif_richard/motifscan_top_dinuc_jaspar2024
FD_DELTA=${FD_RES}/analysis_variant_motif_richard/motifdelta_top_dinuc_jaspar2024

FP_MOTIF=${FD_RES}/analysis_variant_motif_richard/motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl
FP_TBIND=${FD_RES}/analysis_variant_motif_richard/motif_jaspar2024_core_vertebrates_nonredundant.tbind.pkl

mkdir -p "${FD_MSCAN}"
mkdir -p "${FD_DELTA}"

### Execute

In [6]:
### set script
FP_EXE=${FD_EXE}/run_motifdelta_01_scan.sh

### set slurm opt
NUM_CPU=2
#NUM_MEM=40G
#NUM_MEM=20G

LST_OPTS=(
    -A majoroslab
    -p igvf,common
    --cpus-per-task="${NUM_CPU}"
    --chdir="${FD_EXE}"
    --export=ALL,FP_CNF="${FP_CNF}"
    --parsable
)

### loop init
LST_JOBS=()
LST_TAGS=(top01k top10k)

### Loop through I/O
for TXT_TAG in "${LST_TAGS[@]}"; do

    ### set I/O
    TXT_JOB=motifscan.jaspar.${TXT_TAG}
    FP_INP=${FD_BATCH}/variant_closed_gof_bluestarr.flankL35R70.${TXT_TAG}.dinuc.ref.fa.gz
    FP_OUT=${FD_MSCAN}/variant_closed_gof_bluestarr.flankL35R70.${TXT_TAG}.dinuc.npz
    FP_LOG=${FD_LOG}/run.motifscan.jaspar.batch.${TXT_TAG}.txt
    #FP_LOG=${FD_LOG}/run.motifscan.jaspar.batch.${TXT_TAG}.%j.txt
    NUM_FLANK_LEFT=35
    NUM_BATCH_SIZE=2000
    
    ### set memory
    if [[ "${TXT_TAG}" == "top01k" ]]; then
        NUM_MEM=8G
    else
        NUM_MEM=40G
    fi
    
    ### execute
    JOBID=$(sbatch   \
        "${LST_OPTS[@]}" \
        --mem="${NUM_MEM}" \
        --job-name="${TXT_JOB}" \
        --output="${FP_LOG}"    \
        "${FP_EXE}" "${FP_INP}" "${FP_MOTIF}" "${FP_OUT}" "${NUM_FLANK_LEFT}" "${NUM_BATCH_SIZE}"
    )
    echo "Submitted ${TXT_TAG}: ${JOBID}"
    LST_JOBS+=("${JOBID}")
done

Submitted top01k: 42229223
Submitted top10k: 42229224


## Review

In [7]:
for JOBID in "${LST_JOBS[@]}"; do
    sacct_summary.sh "${JOBID}"
    echo
done

===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS                       NodeList 
------------ ------------------------------ ---------- ---------- ---------- ---------- ------------------------------ 
42229223.ba+                          batch  COMPLETED   00:00:18  00:06.916   3173864K                    dcc-core-54 

===== ElapsedRaw =====
ElapsedRaw = 18 sec (0.30 min)

===== MaxRSS =====
MaxRSS = 3.03 GiB

===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS                       NodeList 
------------ ------------------------------ ---------- ---------- ---------- ---------- ------------------------------ 
42229224.ba+                          batch  COMPLETED   00:01:14  01:02.136  30737888K                    dcc-core-54 

===== ElapsedRaw =====
ElapsedRaw = 74 sec (1.23 min)

===== MaxRSS =====
MaxRSS = 29.31 GiB



In [8]:
#cat ${FD_LOG}/run.motifscan.jaspar.batch.${LST_TAGS[0]}.${LST_JOBS[0]}.txt
cat ${FD_LOG}/run.motifscan.jaspar.batch.${LST_TAGS[0]}.txt

Hostname:           dcc-core-54
Slurm Array Index:  NA
Time Stamp:         01-20-26+07:53:24
PWD:                /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts

Loading FASTA sequences...
Loaded 1000 sequences
Load and check complete in 0.01 seconds

Loading motif matrices...
Loaded 879 motifs from /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl
Load and check complete in 0.04 seconds

Setting motif kernel...
Forward kernels shape: (879, 33, 4)
Reverse kernels shape: (879, 33, 4)
Set complete in 0.00 seconds

Running motif scanning...
Scan complete in 2.86 seconds
Output array size (ref+obs+unobs): 1.454 GB

Saving results...
Saved results to /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifscan_top_dinuc_jaspar2024/variant_closed_gof_bluestarr.flankL35R70.top01k.dinuc.npz
Saved complete in 4.66 seconds


Done!
Run Time: 9 seconds



In [9]:
#cat ${FD_LOG}/run.motifscan.jaspar.batch.${LST_TAGS[1]}.${LST_JOBS[1]}.txt
cat ${FD_LOG}/run.motifscan.jaspar.batch.${LST_TAGS[1]}.txt

Hostname:           dcc-core-54
Slurm Array Index:  NA
Time Stamp:         01-20-26+07:53:24
PWD:                /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts

Loading FASTA sequences...
Loaded 10000 sequences
Load and check complete in 0.02 seconds

Loading motif matrices...
Loaded 879 motifs from /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl
Load and check complete in 0.01 seconds

Setting motif kernel...
Forward kernels shape: (879, 33, 4)
Reverse kernels shape: (879, 33, 4)
Set complete in 0.00 seconds

Running motif scanning...
Scan complete in 27.65 seconds
Output array size (ref+obs+unobs): 14.539 GB

Saving results...
Saved results to /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifscan_top_dinuc_jaspar2024/variant_closed_gof_bluestarr.flankL35R70.top10k.dinuc.npz
Saved complete in 36.47 seconds


Done!
Run Time: 1 minutes and 5 sec

## Non-redundant motifs

### Prepare

In [10]:
FD_BATCH=${FD_RES}/analysis_variant_motif_richard/batches_top
FD_MSCAN=${FD_RES}/analysis_variant_motif_richard/motifscan_top_jvierstra_v2.1beta
FD_DELTA=${FD_RES}/analysis_variant_motif_richard/motifdelta_top_jvierstra_v2.1beta

FP_MOTIF=${FD_RES}/analysis_variant_motif_richard/motif_nonredundant_jvierstra_v2.1beta.lods.pkl
FP_TBIND=${FD_RES}/analysis_variant_motif_richard/motif_nonredundant_jvierstra_v2.1beta.tbind.pkl

mkdir -p "${FD_MSCAN}"
mkdir -p "${FD_DELTA}"

### Execute

In [11]:
### set script
FP_EXE=${FD_EXE}/run_motifdelta_01_scan.sh

### set slurm opt
NUM_CPU=2
#NUM_MEM=40G
#NUM_MEM=20G

LST_OPTS=(
    -A majoroslab
    -p igvf,common
    --cpus-per-task="${NUM_CPU}"
    --chdir="${FD_EXE}"
    --export=ALL,FP_CNF="${FP_CNF}"
    --parsable
)

### loop init
LST_JOBS=()
LST_TAGS=(top01k top10k)

### Loop through I/O
for TXT_TAG in ${LST_TAGS[@]}; do

    ### set I/O
    TXT_JOB=motifscan.jvierstra.${TXT_TAG}
    FP_INP=${FD_BATCH}/variant_closed_gof_bluestarr.flankL35R70.${TXT_TAG}.ref.fa.gz
    FP_OUT=${FD_MSCAN}/variant_closed_gof_bluestarr.flankL35R70.${TXT_TAG}.npz
    FP_LOG=${FD_LOG}/run.motifscan.jvierstra.batch.${TXT_TAG}.txt
    #FP_LOG=${FD_LOG}/run.motifscan.jvierstra.batch.${TXT_TAG}.%j.txt
    NUM_FLANK_LEFT=35
    NUM_BATCH_SIZE=2000
    
    ### set memory
    if [[ "${TXT_TAG}" == "top01k" ]]; then
        NUM_MEM=8G
    else
        NUM_MEM=40G
    fi
    
    ### execute
    JOBID=$(sbatch   \
        "${LST_OPTS[@]}" \
        --mem="${NUM_MEM}" \
        --job-name="${TXT_JOB}" \
        --output="${FP_LOG}"    \
        "${FP_EXE}" "${FP_INP}" "${FP_MOTIF}" "${FP_OUT}" "${NUM_FLANK_LEFT}" "${NUM_BATCH_SIZE}"
    )
    echo "Submitted ${TXT_TAG}: ${JOBID}"
    LST_JOBS+=("${JOBID}")
done

Submitted top01k: 42205565
Submitted top10k: 42205566


## Review

In [12]:
for JOBID in "${LST_JOBS[@]}"; do
    sacct_summary.sh "${JOBID}"
    echo
done

===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS 
------------ ------------------------------ ---------- ---------- ---------- ---------- 
42205565.ba+                          batch  COMPLETED   00:00:16  00:04.988   2481932K 

===== ElapsedRaw =====
ElapsedRaw = 16 sec (0.27 min)

===== MaxRSS =====
MaxRSS = 2.37 GiB

===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS 
------------ ------------------------------ ---------- ---------- ---------- ---------- 
42205566.ba+                          batch  COMPLETED   00:06:10  05:31.608  23852880K 

===== ElapsedRaw =====
ElapsedRaw = 370 sec (6.17 min)

===== MaxRSS =====
MaxRSS = 22.75 GiB



In [13]:
#cat ${FD_LOG}/run.motifscan.jvierstra.batch.${LST_TAGS[0]}.${LST_JOBS[0]}.txt
cat ${FD_LOG}/run.motifscan.jvierstra.batch.${LST_TAGS[0]}.txt

Hostname:           dcc-core-02
Slurm Array Index:  NA
Time Stamp:         01-19-26+13:43:13
PWD:                /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts

Loading FASTA sequences...
Loaded 1000 sequences
Load and check complete in 0.01 seconds

Loading motif matrices...
Loaded 637 motifs from /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_nonredundant_jvierstra_v2.1beta.lods.pkl
Load and check complete in 0.01 seconds

Setting motif kernel...
Forward kernels shape: (637, 28, 4)
Reverse kernels shape: (637, 28, 4)
Set complete in 0.00 seconds

Running motif scanning...
Scan complete in 1.71 seconds
Output array size (ref+obs+unobs): 1.125 GB

Saving results...
Saved results to /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifscan_top_jvierstra_v2.1beta/variant_closed_gof_bluestarr.flankL35R70.top01k.npz
Saved complete in 7.08 seconds


Done!
Run Time: 9 seconds



In [14]:
#cat ${FD_LOG}/run.motifscan.jvierstra.batch.${LST_TAGS[1]}.${LST_JOBS[1]}.txt
cat ${FD_LOG}/run.motifscan.jvierstra.batch.${LST_TAGS[1]}.txt

Hostname:           dcc-comp-02
Slurm Array Index:  NA
Time Stamp:         01-19-26+13:43:16
PWD:                /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts

Loading FASTA sequences...
Loaded 10000 sequences
Load and check complete in 0.04 seconds

Loading motif matrices...
Loaded 637 motifs from /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_nonredundant_jvierstra_v2.1beta.lods.pkl
Load and check complete in 0.01 seconds

Setting motif kernel...
Forward kernels shape: (637, 28, 4)
Reverse kernels shape: (637, 28, 4)
Set complete in 0.00 seconds

Running motif scanning...
Scan complete in 310.47 seconds
Output array size (ref+obs+unobs): 11.248 GB

Saving results...
Saved results to /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifscan_top_jvierstra_v2.1beta/variant_closed_gof_bluestarr.flankL35R70.top10k.npz
Saved complete in 47.23 seconds


Done!
Run Time: 6 minutes and 0 seconds



## Review (Batch size = 1000; float32)

In [17]:
for JOBID in "${LST_JOBS[@]}"; do
    sacct_summary.sh "${JOBID}"
    echo
done

===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS 
------------ ------------------------------ ---------- ---------- ---------- ---------- 
42097497.ba+                          batch  COMPLETED   00:00:17  00:07.212   3168664K 

===== ElapsedRaw =====
ElapsedRaw = 17 sec (0.28 min)

===== MaxRSS =====
MaxRSS = 3.02 GiB

===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS 
------------ ------------------------------ ---------- ---------- ---------- ---------- 
42097498.ba+                          batch  COMPLETED   00:01:23  01:03.206  30741024K 

===== ElapsedRaw =====
ElapsedRaw = 83 sec (1.38 min)

===== MaxRSS =====
MaxRSS = 29.32 GiB



In [18]:
cat ${FD_LOG}/run.motifscan.jaspar.batch.${LST_TAGS[0]}.${LST_JOBS[0]}.txt

Hostname:           dcc-allenlab-01
Slurm Array Index:  NA
Time Stamp:         01-15-26+14:57:30
PWD:                /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts

Loading FASTA sequences...
Loaded 1000 sequences
Load and check complete in 0.01 seconds

Loading motif matrices...
Loaded 879 motifs from /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl
Load and check complete in 0.01 seconds

Setting motif kernel...
Forward kernels shape: (879, 33, 4)
Reverse kernels shape: (879, 33, 4)
Set complete in 0.01 seconds

Running motif scanning...
Scan complete in 3.17 seconds
Output array size (ref+obs+unobs): 1.454 GB

Saving results...
Saved results to /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifscan_top_jaspar2024/variant_closed_gof_bluestarr_flankL35R70_top01k.npz
Saved complete in 6.05 seconds


Done!
Run Time: 9 seconds



In [19]:
cat ${FD_LOG}/run.motifscan.jaspar.batch.${LST_TAGS[1]}.${LST_JOBS[1]}.txt

Hostname:           dcc-allenlab-01
Slurm Array Index:  NA
Time Stamp:         01-15-26+14:57:29
PWD:                /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts

Loading FASTA sequences...
Loaded 10000 sequences
Load and check complete in 0.04 seconds

Loading motif matrices...
Loaded 879 motifs from /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl
Load and check complete in 0.01 seconds

Setting motif kernel...
Forward kernels shape: (879, 33, 4)
Reverse kernels shape: (879, 33, 4)
Set complete in 0.01 seconds

Running motif scanning...
Scan complete in 33.20 seconds
Output array size (ref+obs+unobs): 14.539 GB

Saving results...
Saved results to /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifscan_top_jaspar2024/variant_closed_gof_bluestarr_flankL35R70_top10k.npz
Saved complete in 41.38 seconds


Done!
Run Time: 1 minutes and 16 seconds



## Review (Npz no index; float32)

In [33]:
for JOBID in "${LST_JOBS[@]}"; do
    sacct_summary.sh "${JOBID}"
    echo
done

===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS 
------------ ------------------------------ ---------- ---------- ---------- ---------- 
41952368.ba+                          batch  COMPLETED   00:00:14  00:07.314   3188120K 

===== ElapsedRaw =====
ElapsedRaw = 14 sec (0.23 min)

===== MaxRSS =====
MaxRSS = 3.04 GiB

===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS 
------------ ------------------------------ ---------- ---------- ---------- ---------- 
41952369.ba+                          batch  COMPLETED   00:01:13  01:03.197  30756888K 

===== ElapsedRaw =====
ElapsedRaw = 73 sec (1.22 min)

===== MaxRSS =====
MaxRSS = 29.33 GiB



In [34]:
cat ${FD_LOG}/run.motifscan.jaspar.batch.${LST_TAGS[0]}.${LST_JOBS[0]}.txt

Hostname:           dcc-allenlab-01
Slurm Array Index:  NA
Time Stamp:         01-14-26+16:33:40
PWD:                /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts

Loading FASTA sequences...
Loaded 1000 sequences
Load and check complete in 0.01 seconds

Loading motif matrices...
Loaded 879 motifs from /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl
Load and check complete in 0.01 seconds

Setting motif kernel...
Forward kernels shape: (879, 33, 4)
Reverse kernels shape: (879, 33, 4)
Set complete in 0.01 seconds

Running motif scanning...
Scan complete in 2.87 seconds
Estimated memory use (ref+obs+unobs): 1.454 GB

Saving results...
Saved results to /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifscan_top_jaspar2024/variant_closed_gof_bluestarr_flankL35R70_top01k.npz
Saved complete in 4.33 seconds


Done!
Run Time: 8 seconds



In [35]:
cat ${FD_LOG}/run.motifscan.jaspar.batch.${LST_TAGS[1]}.${LST_JOBS[1]}.txt

Hostname:           dcc-allenlab-01
Slurm Array Index:  NA
Time Stamp:         01-14-26+16:33:41
PWD:                /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts

Loading FASTA sequences...
Loaded 10000 sequences
Load and check complete in 0.04 seconds

Loading motif matrices...
Loaded 879 motifs from /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl
Load and check complete in 0.01 seconds

Setting motif kernel...
Forward kernels shape: (879, 33, 4)
Reverse kernels shape: (879, 33, 4)
Set complete in 0.01 seconds

Running motif scanning...
Scan complete in 31.56 seconds
Estimated memory use (ref+obs+unobs): 14.539 GB

Saving results...
Saved results to /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifscan_top_jaspar2024/variant_closed_gof_bluestarr_flankL35R70_top10k.npz
Saved complete in 33.86 seconds


Done!
Run Time: 1 minutes and 6 seconds


## Review (Npz no index)

In [29]:
for JOBID in "${LST_JOBS[@]}"; do
    sacct_summary.sh "${JOBID}"
    echo
done

===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS 
------------ ------------------------------ ---------- ---------- ---------- ---------- 
41951744.ba+                          batch  COMPLETED   00:00:15  00:07.340   3183536K 

===== ElapsedRaw =====
ElapsedRaw = 15 sec (0.25 min)

===== MaxRSS =====
MaxRSS = 3.04 GiB

===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS 
------------ ------------------------------ ---------- ---------- ---------- ---------- 
41951745.ba+                          batch  COMPLETED   00:01:12  01:02.635  30752088K 

===== ElapsedRaw =====
ElapsedRaw = 72 sec (1.20 min)

===== MaxRSS =====
MaxRSS = 29.33 GiB



In [30]:
cat ${FD_LOG}/run.motifscan.jaspar.batch.${LST_TAGS[0]}.${LST_JOBS[0]}.txt

Hostname:           dcc-allenlab-01
Slurm Array Index:  NA
Time Stamp:         01-14-26+16:29:55
PWD:                /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts

Loading FASTA sequences...
Loaded 1000 sequences
Load and check complete in 0.01 seconds

Loading motif matrices...
Loaded 879 motifs from /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl
Load and check complete in 0.01 seconds

Setting motif kernel...
Forward kernels shape: (879, 33, 4)
Reverse kernels shape: (879, 33, 4)
Set complete in 0.01 seconds

Running motif scanning...
Scan complete in 2.81 seconds
Estimated memory use (ref+obs+unobs): 1.454 GB

Saving results...
Saved results to /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifscan_top_jaspar2024/variant_closed_gof_bluestarr_flankL35R70_top01k.npz
Saved complete in 4.62 seconds


Done!
Run Time: 8 seconds



In [31]:
cat ${FD_LOG}/run.motifscan.jaspar.batch.${LST_TAGS[1]}.${LST_JOBS[1]}.txt

Hostname:           dcc-allenlab-01
Slurm Array Index:  NA
Time Stamp:         01-14-26+16:29:55
PWD:                /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts

Loading FASTA sequences...
Loaded 10000 sequences
Load and check complete in 0.04 seconds

Loading motif matrices...
Loaded 879 motifs from /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl
Load and check complete in 0.01 seconds

Setting motif kernel...
Forward kernels shape: (879, 33, 4)
Reverse kernels shape: (879, 33, 4)
Set complete in 0.01 seconds

Running motif scanning...
Scan complete in 31.16 seconds
Estimated memory use (ref+obs+unobs): 14.539 GB

Saving results...
Saved results to /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifscan_top_jaspar2024/variant_closed_gof_bluestarr_flankL35R70_top10k.npz
Saved complete in 33.26 seconds


Done!
Run Time: 1 minutes and 5 seconds


## Review (Npy)

In [22]:
for JOBID in "${LST_JOBS[@]}"; do
    sacct_summary.sh "${JOBID}"
    echo
done

===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS 
------------ ------------------------------ ---------- ---------- ---------- ---------- 
41949517.ba+                          batch  COMPLETED   00:00:21  00:12.084   3660796K 

===== ElapsedRaw =====
ElapsedRaw = 21 sec (0.35 min)

===== MaxRSS =====
MaxRSS = 3.49 GiB

===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS 
------------ ------------------------------ ---------- ---------- ---------- ---------- 
41949518.ba+                          batch  COMPLETED   00:01:57  01:49.206  35813564K 

===== ElapsedRaw =====
ElapsedRaw = 117 sec (1.95 min)

===== MaxRSS =====
MaxRSS = 34.15 GiB



In [25]:
cat ${FD_LOG}/run.motifscan.jaspar.batch.${LST_TAGS[0]}.${LST_JOBS[0]}.txt

Hostname:           dcc-allenlab-01
Slurm Array Index:  NA
Time Stamp:         01-14-26+16:15:31
PWD:                /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts

Loading FASTA sequences...
Loaded 1000 sequences
Load and check complete in 0.01 seconds

Loading motif matrices...
Loaded 879 motifs from /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl
Load and check complete in 0.01 seconds

Setting motif kernel...
Forward kernels shape: (879, 33, 4)
Reverse kernels shape: (879, 33, 4)
Set complete in 0.01 seconds

Running motif scanning...
Scan complete in 2.85 seconds
Estimated memory use (ref+obs+unobs): 1.454 GB

Saving results...
Saved results to /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifscan_top_jaspar2024/variant_closed_gof_bluestarr_flankL35R70_top01k_ref.npy
Saved complete in 9.47 seconds


Done!
Run Time: 13 seconds



In [26]:
cat ${FD_LOG}/run.motifscan.jaspar.batch.${LST_TAGS[1]}.${LST_JOBS[1]}.txt

Hostname:           dcc-allenlab-01
Slurm Array Index:  NA
Time Stamp:         01-14-26+16:15:31
PWD:                /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts

Loading FASTA sequences...
Loaded 10000 sequences
Load and check complete in 0.04 seconds

Loading motif matrices...
Loaded 879 motifs from /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl
Load and check complete in 0.01 seconds

Setting motif kernel...
Forward kernels shape: (879, 33, 4)
Reverse kernels shape: (879, 33, 4)
Set complete in 0.01 seconds

Running motif scanning...
Scan complete in 31.13 seconds
Estimated memory use (ref+obs+unobs): 14.539 GB

Saving results...
Saved results to /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifscan_top_jaspar2024/variant_closed_gof_bluestarr_flankL35R70_top10k_ref.npy
Saved complete in 77.44 seconds


Done!
Run Time: 1 minutes and 49 sec

## Review (Original)

In [18]:
for JOBID in "${LST_JOBS[@]}"; do
    sacct_summary.sh "${JOBID}"
    echo
done

===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS 
------------ ------------------------------ ---------- ---------- ---------- ---------- 
41946511.ba+                          batch  COMPLETED   00:00:16  00:07.528   3184312K 

===== ElapsedRaw =====
ElapsedRaw = 16 sec (0.27 min)

===== MaxRSS =====
MaxRSS = 3.04 GiB

===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS 
------------ ------------------------------ ---------- ---------- ---------- ---------- 
41946512.ba+                          batch  COMPLETED   00:01:50  01:05.462  30754864K 

===== ElapsedRaw =====
ElapsedRaw = 110 sec (1.83 min)

===== MaxRSS =====
MaxRSS = 29.33 GiB



In [19]:
cat ${FD_LOG}/run.motifscan.jaspar.batch.${LST_TAGS[0]}.${LST_JOBS[0]}.txt

Hostname:           dcc-allenlab-01
Slurm Array Index:  NA
Time Stamp:         01-14-26+15:51:30
PWD:                /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts

Loading FASTA sequences...
Loaded 1000 sequences
Load and check complete in 0.01 seconds

Loading motif matrices...
Loaded 879 motifs from /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl
Load and check complete in 0.01 seconds

Setting motif kernel...
Forward kernels shape: (879, 33, 4)
Reverse kernels shape: (879, 33, 4)
Set complete in 0.01 seconds

Running motif scanning...
Scan complete in 2.83 seconds
Estimated memory use (ref+obs+unobs): 1.454 GB

Saving results...
Saved results to /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifscan_top_jaspar2024/variant_closed_gof_bluestarr_flankL35R70_top01k_ref.npz
Saved complete in 5.20 seconds


Done!
Run Time: 9 seconds



In [20]:
cat ${FD_LOG}/run.motifscan.jaspar.batch.${LST_TAGS[1]}.${LST_JOBS[1]}.txt

Hostname:           dcc-allenlab-01
Slurm Array Index:  NA
Time Stamp:         01-14-26+15:51:30
PWD:                /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts

Loading FASTA sequences...
Loaded 10000 sequences
Load and check complete in 0.03 seconds

Loading motif matrices...
Loaded 879 motifs from /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl
Load and check complete in 0.01 seconds

Setting motif kernel...
Forward kernels shape: (879, 33, 4)
Reverse kernels shape: (879, 33, 4)
Set complete in 0.01 seconds

Running motif scanning...
Scan complete in 31.28 seconds
Estimated memory use (ref+obs+unobs): 14.539 GB

Saving results...
Saved results to /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifscan_top_jaspar2024/variant_closed_gof_bluestarr_flankL35R70_top10k_ref.npz
Saved complete in 70.46 seconds


Done!
Run Time: 1 minutes and 43 sec

In [28]:
### set script
FP_EXE=${FD_EXE}/run_motifdelta_01_scan.py

### set resource
NUM_CPU=2
NUM_MEM=20G

SLURM_OPTS=(
  -A majoroslab
  -p igvf
  --job-name=motifscan_batch_top_jaspar2024
  --cpus-per-task="${NUM_CPU}"
  --mem="${NUM_MEM}"
  --chdir="${FD_EXE}"
  --export=ALL,FP_CNF="${FP_CNF}"
  --parsable
)

### I/O
FP_INP=${FD_BATCH}/variant_closed_gof_bluestarr_flankL35R70_top01k_ref.fa.gz
FP_OUT=${FD_MSCAN}/variant_closed_gof_bluestarr_flankL35R70_top01k_ref.npy
FP_LOG=${FD_LOG}/run.motifscan.batch_top.jaspar.top01k.%j.txt

### execute
SLURM_JOBID=$(sbatch \
    "${SLURM_OPTS[@]}" \
    --output="${FP_LOG}" \
    <<EOF
#!/bin/bash
set -euo pipefail

### init
timer_start=\$(date +%s)
source "${FP_CNF}"

### print start message
echo "Hostname:   \$(hostname)"
echo "Time Stamp: \$(date +'%m-%d-%y+%T')"
echo "PWD:    \$(pwd)"
echo "FP_APP: ${FP_APP}"
echo "FD_EXE: ${FD_EXE}"
echo "PYTHONPATH (host): \${PYTHONPATH:-<empty>}"
echo

### execute
echo "=== Run main script ==="
${FP_APP} python ${FP_EXE} \
    --txt_fpath_fasta  "${FP_INP}" \
    --txt_fpath_motif  "${FP_MOTIF}" \
    --txt_fpath_output "${FP_OUT}" \
    --num_flank_left   35

### print end message
timer=\$(date +%s)
runtime=\$(( timer - timer_start ))
echo
echo 'Done!'
echo "Run Time: \$(displaytime \${runtime})"
EOF
)

echo "Submitted job: ${SLURM_JOBID}"

Submitted job: 41784707


### Review

In [29]:
sacct_summary.sh ${SLURM_JOBID}

===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS 
------------ ------------------------------ ---------- ---------- ---------- ---------- 
41784707.ba+                          batch     FAILED   00:00:09  00:04.703   2677264K 

===== ElapsedRaw =====
ElapsedRaw = 9 sec (0.15 min)

===== MaxRSS =====
MaxRSS = 2.55 GiB


In [30]:
cat ${FD_LOG}/run.motifscan.batch_top.jaspar.top01k.${SLURM_JOBID}.txt

Hostname:   dcc-allenlab-01
Time Stamp: 01-13-26+15:13:46
PWD:    /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts
FP_APP: /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts/run_script.sh
FD_EXE: /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts
PYTHONPATH (host): <empty>

=== Run main script ===
Loading FASTA sequences...
Loaded 1000 sequences
Load and check complete in 0.01 seconds

Loading motif matrices...
Loaded 879 motifs from /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl
Load and check complete in 0.01 seconds

Setting motif kernel...
Forward kernels shape: (879, 33, 4)
Reverse kernels shape: (879, 33, 4)
Set complete in 0.01 seconds

Running motif scanning...
Scan complete in 2.80 seconds
Estimated memory use (ref+obs+unobs): 1.454 GB

Saving results...
Traceback (most recent call last):
  File "/hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts/run_motifdelta_01_s

In [9]:
cat ${FD_LOG}/run.motifscan.batch_top.jaspar.top01k.${SLURM_JOBID}.txt

Hostname:   dcc-allenlab-01
Time Stamp: 01-13-26+12:07:13
PWD:    /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts
FP_APP: /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts/run_script.sh
FD_EXE: /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts
PYTHONPATH (host): <empty>

=== Run main script ===
Loading FASTA sequences...
Loaded 1000 sequences
106

Done!
Run Time: 1 seconds


In [ ]:
### set resource
NUM_CPU=5
NUM_MEM=20G

### loop each chunk and run motifdelta
for idx in $(seq 1 20); do

    ### prepare
    CHUNK=$(printf "chunk%03d" "${idx}")
    PREFIX=variant_closed_gof_bluestarr_flankL35R70_pilot_${CHUNK}
    echo "Running motif scan for ${PREFIX}..."

    ### execute
    FP_EXE=${FD_EXE}/run_motifdelta_01_scan.py
    FN_LOG=run_motifdelta_scan_batch_pilot_jaspar_${CHUNK}.txt
    FP_LOG=${FD_LOG}/${FN_LOG}
    echo '${FD_LOG}'/"${FN_LOG}"
    
    sbatch \
        -A majoroslab \
        -p igvf,common \
        --cpus-per-task ${NUM_CPU} \
        --mem    ${NUM_MEM} \
        --output ${FP_LOG}  \
        --chdir  ${FD_EXE}  \
        ${FP_APP} python ${FP_EXE} \
            --txt_fpath_fasta_ref  ${FD_BATCH}/${PREFIX}_ref.fa \
            --txt_fpath_fasta_obs  ${FD_BATCH}/${PREFIX}_obs.fa \
            --txt_fpath_fasta_ubs  ${FD_BATCH}/${PREFIX}_unobs.fa \
            --txt_fpath_motif      ${FP_MOTIF} \
            --txt_fpath_output     ${FD_DELTA}/${PREFIX}_scan.npz
done

In [4]:
ls ${FD_RES}/analysis_variant_motif_richard/batches_pilot

variant_closed_gof_bluestarr_flankL35R70_pilot_chunk001_obs.fa
variant_closed_gof_bluestarr_flankL35R70_pilot_chunk001_ref.fa
variant_closed_gof_bluestarr_flankL35R70_pilot_chunk001_unobs.fa
variant_closed_gof_bluestarr_flankL35R70_pilot_chunk002_obs.fa
variant_closed_gof_bluestarr_flankL35R70_pilot_chunk002_ref.fa
variant_closed_gof_bluestarr_flankL35R70_pilot_chunk002_unobs.fa
variant_closed_gof_bluestarr_flankL35R70_pilot_chunk003_obs.fa
variant_closed_gof_bluestarr_flankL35R70_pilot_chunk003_ref.fa
variant_closed_gof_bluestarr_flankL35R70_pilot_chunk003_unobs.fa
variant_closed_gof_bluestarr_flankL35R70_pilot_chunk004_obs.fa
variant_closed_gof_bluestarr_flankL35R70_pilot_chunk004_ref.fa
variant_closed_gof_bluestarr_flankL35R70_pilot_chunk004_unobs.fa
variant_closed_gof_bluestarr_flankL35R70_pilot_chunk005_obs.fa
variant_closed_gof_bluestarr_flankL35R70_pilot_chunk005_ref.fa
variant_closed_gof_bluestarr_flankL35R70_pilot_chunk005_unobs.fa
variant_closed_gof_bluestarr_flankL35R70_pilo

## Execute

### Jaspar2024

In [12]:
FD_BATCH=${FD_RES}/analysis_variant_motif_richard/batches_pilot
FP_MOTIF=${FD_RES}/analysis_variant_motif_richard/motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl
FP_MODEL=${FD_RES}/analysis_variant_motif_richard/motif_jaspar2024_core_vertebrates_nonredundant.pmap.pkl
FD_DELTA=${FD_RES}/analysis_variant_motif_richard/motifdelta_pilot_jaspar2024
#mkdir -p "${FD_DELTA}"

In [13]:
### set resource
NUM_CPU=5
NUM_MEM=20G

### loop each chunk and run motifdelta
for idx in $(seq 1 20); do

    ### prepare
    CHUNK=$(printf "chunk%03d" "${idx}")
    PREFIX=variant_closed_gof_bluestarr_flankL35R70_pilot_${CHUNK}
    echo "Running motif scan for ${PREFIX}..."

    ### execute
    FP_EXE=${FD_EXE}/run_motifdelta_01_scan.py
    FN_LOG=run_motifdelta_scan_batch_pilot_jaspar_${CHUNK}.txt
    FP_LOG=${FD_LOG}/${FN_LOG}
    echo '${FD_LOG}'/"${FN_LOG}"
    
    sbatch \
        -A majoroslab \
        -p igvf,common \
        --cpus-per-task ${NUM_CPU} \
        --mem    ${NUM_MEM} \
        --output ${FP_LOG}  \
        --chdir  ${FD_EXE}  \
        ${FP_APP} python ${FP_EXE} \
            --txt_fpath_fasta_ref  ${FD_BATCH}/${PREFIX}_ref.fa \
            --txt_fpath_fasta_obs  ${FD_BATCH}/${PREFIX}_obs.fa \
            --txt_fpath_fasta_ubs  ${FD_BATCH}/${PREFIX}_unobs.fa \
            --txt_fpath_motif      ${FP_MOTIF} \
            --txt_fpath_output     ${FD_DELTA}/${PREFIX}_scan.npz
done

Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk001...
${FD_LOG}/run_motifdelta_scan_batch_pilot_jaspar_chunk001.txt
Submitted batch job 40271066
Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk002...
${FD_LOG}/run_motifdelta_scan_batch_pilot_jaspar_chunk002.txt
Submitted batch job 40271067
Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk003...
${FD_LOG}/run_motifdelta_scan_batch_pilot_jaspar_chunk003.txt
Submitted batch job 40271068
Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk004...
${FD_LOG}/run_motifdelta_scan_batch_pilot_jaspar_chunk004.txt
Submitted batch job 40271069
Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk005...
${FD_LOG}/run_motifdelta_scan_batch_pilot_jaspar_chunk005.txt
Submitted batch job 40271070
Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk006...
${FD_LOG}/run_motifdelta_scan_batch_pilot_jaspar_chun

### jvierstra non-redundant motif

In [21]:
FD_BATCH=${FD_RES}/analysis_variant_motif_richard/batches_pilot
FP_MOTIF=${FD_RES}/analysis_variant_motif_richard/motif_nonredundant_jvierstra_v2.0beta.lods.pkl
FP_MODEL=${FD_RES}/analysis_variant_motif_richard/motif_nonredundant_jvierstra_v2.0beta.pmap.pkl
FD_DELTA=${FD_RES}/analysis_variant_motif_richard/motifdelta_pilot_jvierstra_v2.0beta
#mkdir -p "${FD_DELTA}"

In [22]:
### set resource
NUM_CPU=5
NUM_MEM=20G

### loop each chunk and run motifdelta
for idx in $(seq 1 20); do

    ### prepare
    CHUNK=$(printf "chunk%03d" "${idx}")
    PREFIX=variant_closed_gof_bluestarr_flankL35R70_pilot_${CHUNK}
    echo "Running motif scan for ${PREFIX}..."

    ### execute
    FP_EXE=${FD_EXE}/run_motifdelta_01_scan.py
    FN_LOG=run_motifdelta_scan_batch_pilot_jvierstra_${CHUNK}.txt
    FP_LOG=${FD_LOG}/${FN_LOG}
    echo '${FD_LOG}'/"${FN_LOG}"
    
    sbatch \
        -A majoroslab \
        -p igvf,common \
        --cpus-per-task ${NUM_CPU} \
        --mem    ${NUM_MEM} \
        --output ${FP_LOG}  \
        --chdir  ${FD_EXE}  \
        ${FP_APP} python ${FP_EXE} \
            --txt_fpath_fasta_ref  ${FD_BATCH}/${PREFIX}_ref.fa \
            --txt_fpath_fasta_obs  ${FD_BATCH}/${PREFIX}_obs.fa \
            --txt_fpath_fasta_ubs  ${FD_BATCH}/${PREFIX}_unobs.fa \
            --txt_fpath_motif      ${FP_MOTIF} \
            --txt_fpath_output     ${FD_DELTA}/${PREFIX}_scan.npz
done

Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk001...
${FD_LOG}/run_motifdelta_scan_batch_pilot_jvierstra_chunk001.txt
Submitted batch job 40271430
Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk002...
${FD_LOG}/run_motifdelta_scan_batch_pilot_jvierstra_chunk002.txt
Submitted batch job 40271431
Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk003...
${FD_LOG}/run_motifdelta_scan_batch_pilot_jvierstra_chunk003.txt
Submitted batch job 40271432
Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk004...
${FD_LOG}/run_motifdelta_scan_batch_pilot_jvierstra_chunk004.txt
Submitted batch job 40271433
Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk005...
${FD_LOG}/run_motifdelta_scan_batch_pilot_jvierstra_chunk005.txt
Submitted batch job 40271434
Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk006...
${FD_LOG}/run_motifdelta_scan_batch_pi

## Review

### JASPAR 2024

In [14]:
cat ${FD_LOG}/run_motifdelta_scan_batch_pilot_jaspar_chunk001.txt

Loading FASTA sequences...
Loaded 5000 sequences
Load and check complete in 0.06 seconds

Loading motif matrices...
Loaded 879 motifs from /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl
Load and check complete in 0.01 seconds

Setting motif kernel...
Forward kernels shape: (879, 33, 4)
Reverse kernels shape: (879, 33, 4)
Set complete in 0.01 seconds

Running motif scanning...
Scan complete in 14.34 seconds
Estimated memory use (ref+obs+unobs): 7.269 GB

Saving results...
Saved results to /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifdelta_pilot_jaspar2024/variant_closed_gof_bluestarr_flankL35R70_pilot_chunk001_scan.npz
Saved complete in 26.28 seconds

Done.


In [17]:
JOBIDS=40271066
sacct_summary.sh ${JOBIDS}

===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS 
------------ ------------------------------ ---------- ---------- ---------- ---------- 
40271066.ba+                          batch  COMPLETED   00:00:42  00:38.067     15112M 

===== ElapsedRaw =====
ElapsedRaw = 42 sec (0.70 min)

===== MaxRSS =====
MaxRSS = 14.76 GiB


In [16]:
cat ${FD_LOG}/run_motifdelta_scan_batch_pilot_jaspar_chunk020.txt

Loading FASTA sequences...
Loaded 5000 sequences
Load and check complete in 0.08 seconds

Loading motif matrices...
Loaded 879 motifs from /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl
Load and check complete in 0.01 seconds

Setting motif kernel...
Forward kernels shape: (879, 33, 4)
Reverse kernels shape: (879, 33, 4)
Set complete in 0.01 seconds

Running motif scanning...
Scan complete in 91.04 seconds
Estimated memory use (ref+obs+unobs): 7.269 GB

Saving results...
Saved results to /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifdelta_pilot_jaspar2024/variant_closed_gof_bluestarr_flankL35R70_pilot_chunk020_scan.npz
Saved complete in 33.15 seconds

Done.


In [18]:
JOBIDS=40271085
sacct_summary.sh ${JOBIDS}

===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS 
------------ ------------------------------ ---------- ---------- ---------- ---------- 
40271085.ba+                          batch  COMPLETED   00:02:08  01:58.141  15469988K 

===== ElapsedRaw =====
ElapsedRaw = 128 sec (2.13 min)

===== MaxRSS =====
MaxRSS = 14.75 GiB


In [20]:
JOBIDS=$(seq 40271066 40271085 | paste -sd' ')
sacct_summary.sh ${JOBIDS}

Detected multiple job IDs (20). Running batch summary.
40271066,40271067,40271068,40271069,40271070,40271071,40271072,40271073,40271074,40271075,40271076,40271077,40271078,40271079,40271080,40271081,40271082,40271083,40271084,40271085
===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS 
------------ ------------------------------ ---------- ---------- ---------- ---------- 
40271066.ba+                          batch  COMPLETED   00:00:42  00:38.067     15112M 
40271067.ba+                          batch  COMPLETED   00:00:41  00:37.990  15477028K 
40271068.ba+                          batch  COMPLETED   00:02:03  00:46.506  15473804K 
40271069.ba+                          batch  COMPLETED   00:02:02  00:45.630  15478112K 
40271070.ba+                          batch  COMPLETED   00:01:49  00:43.629  15474344K 
40271071.ba+                          batch  COMPLETED   00:02:03  00:46.290  15476036K 
40271072.ba+         

### jvierstra non-redundant motif

In [24]:
JOBIDS=$(seq 40271430 40271449 | paste -sd' ')
sacct_summary.sh ${JOBIDS}

Detected multiple job IDs (20). Running batch summary.
40271430,40271431,40271432,40271433,40271434,40271435,40271436,40271437,40271438,40271439,40271440,40271441,40271442,40271443,40271444,40271445,40271446,40271447,40271448,40271449
===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS 
------------ ------------------------------ ---------- ---------- ---------- ---------- 
40271430.ba+                          batch  COMPLETED   00:01:45  00:42.322  13057384K 
40271431.ba+                          batch  COMPLETED   00:01:44  00:43.025  13057000K 
40271432.ba+                          batch  COMPLETED   00:01:45  00:41.517  13053844K 
40271433.ba+                          batch  COMPLETED   00:01:45  00:40.800  13055200K 
40271434.ba+                          batch  COMPLETED   00:01:42  00:41.353  13058152K 
40271435.ba+                          batch  COMPLETED   00:01:44  00:41.716  13053640K 
40271436.ba+         

```
for JobID in $(seq 40150927 40150948); do
    sacct -j ${JobID} --format=JobID,JobName%30,State,Elapsed,TotalCPU,MaxRSS \
    | grep '\.ba'
done
```

In [16]:
JOBIDS=40166308
sacct_summary.sh ${JOBIDS}

===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS 
------------ ------------------------------ ---------- ---------- ---------- ---------- 
40166308.ba+                          batch  COMPLETED   00:01:58  00:47.316  15475320K 

===== ElapsedRaw =====
ElapsedRaw = 118 sec (1.97 min)

===== MaxRSS =====
MaxRSS = 14.76 GiB


In [18]:
JOBIDS=$(seq 40166308 40166328 | paste -sd' ')
echo ${JOBIDS}

40166308 40166309 40166310 40166311 40166312 40166313 40166314 40166315 40166316 40166317 40166318 40166319 40166320 40166321 40166322 40166323 40166324 40166325 40166326 40166327 40166328


In [24]:
JOBIDS=$(seq 40166308 40166328 | paste -sd' ')
JOBID=$(echo ${JOBIDS[0]} | tr ' ' ',')
echo ${JOBID}

40166308,40166309,40166310,40166311,40166312,40166313,40166314,40166315,40166316,40166317,40166318,40166319,40166320,40166321,40166322,40166323,40166324,40166325,40166326,40166327,40166328


In [23]:
JOBIDS=$(seq 40166308 40166328 | paste -sd' ')
sacct_summary.sh ${JOBIDS}

Detected multiple job IDs (21). Running batch summary.
40166308
===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS 
------------ ------------------------------ ---------- ---------- ---------- ---------- 
40166308.ba+                          batch  COMPLETED   00:01:58  00:47.316  15475320K 

===== ElapsedRaw =====
n=1  min=118 sec (2.0 min)  mean=118.0 sec (2.0 min)  max=118 sec (2.0 min)

===== MaxRSS =====
n=1  min=14.76 GiB  mean=14.76 GiB  max=14.76 GiB


In [8]:
JobIDS=$(seq 40166308 40166328 | paste -sd, -)
echo ${JobIDS}

40166308,40166309,40166310,40166311,40166312,40166313,40166314,40166315,40166316,40166317,40166318,40166319,40166320,40166321,40166322,40166323,40166324,40166325,40166326,40166327,40166328


In [26]:
JOBIDS=$(seq 40166308 40166328 | paste -sd' ')
sacct_summary_batch.sh ${JobIDS}

===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS 
------------ ------------------------------ ---------- ---------- ---------- ---------- 
40166308.ba+                          batch  COMPLETED   00:01:58  00:47.316  15475320K 
40166309.ba+                          batch  COMPLETED   00:01:56  00:47.819  15476952K 
40166310.ba+                          batch  COMPLETED   00:01:54  00:43.897  15470564K 
40166311.ba+                          batch  COMPLETED   00:01:57  00:47.623  15472516K 
40166312.ba+                          batch  COMPLETED   00:01:57  00:47.697  15472272K 
40166313.ba+                          batch  COMPLETED   00:01:58  00:49.020  15473072K 
40166314.ba+                          batch  COMPLETED   00:01:56  00:46.472  15472940K 
40166315.ba+                          batch  COMPLETED   00:01:53  00:43.290  15474320K 
40166316.ba+                          batch  COMPLETED   00:00:46  00:41.182  

In [6]:
JobIDS=$(seq 40166308 40166328 | paste -sd, -)
sacct_summary_batch.sh ${JobIDS}

===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS 
------------ ------------------------------ ---------- ---------- ---------- ---------- 
40166308.ba+                          batch  COMPLETED   00:01:58  00:47.316  15475320K 
40166309.ba+                          batch  COMPLETED   00:01:56  00:47.819  15476952K 
40166310.ba+                          batch  COMPLETED   00:01:54  00:43.897  15470564K 
40166311.ba+                          batch  COMPLETED   00:01:57  00:47.623  15472516K 
40166312.ba+                          batch  COMPLETED   00:01:57  00:47.697  15472272K 
40166313.ba+                          batch  COMPLETED   00:01:58  00:49.020  15473072K 
40166314.ba+                          batch  COMPLETED   00:01:56  00:46.472  15472940K 
40166315.ba+                          batch  COMPLETED   00:01:53  00:43.290  15474320K 
40166316.ba+                          batch  COMPLETED   00:00:46  00:41.182  

In [7]:
JobIDS=40166308
sacct_summary_single.sh ${JobIDS}

===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS 
------------ ------------------------------ ---------- ---------- ---------- ---------- 
40166308.ba+                          batch  COMPLETED   00:01:58  00:47.316  15475320K 

===== ElapsedRaw =====
ElapsedRaw = 118 sec (1.97 min)

===== MaxRSS =====
MaxRSS = 14.76 GiB


In [22]:
JobIDS=$(seq 40166308 40166328 | paste -sd, -)
echo ${JobIDS}
sacct -j ${JobIDS} --format=JobID,JobName%30,State,Elapsed,TotalCPU,MaxRSS \
| grep '\.ba'

sacct -j ${JobIDS} --format=JobID,ElapsedRaw \
| grep '\.ba' \
| awk '
    {
        t = $2                          # ElapsedRaw in seconds
        if (NR == 1 || t < min) min = t # get min
        if (NR == 1 || t > max) max = t # get max 
        sum += t                        # get sum
        n++                             # get number of row
    }
    END {
        mean = (n > 0) ? sum / n : 0 
        printf "n=%d  min=%d sec (%.2f min)  mean=%.2f sec (%.2f min)  max=%d sec (%.2f min)\n",
                n, min, min/60, mean, mean/60, max, max/60
    }
'

sacct -j ${JobIDS} --format=JobID,MaxRSS \
| grep '\.ba' \
| awk '
{
    raw = $2             # e.g. "15480360K"
    unit = substr(raw, length(raw), 1)
    gsub(/[^0-9]/, "", raw)
    val = raw + 0        # numeric value

    # Convert to GiB
    if (unit == "K") {
        gib = val / 1024 / 1024      # K -> GiB
    } else if (unit == "M") {
        gib = val / 1024             # MiB -> GiB
    } else if (unit == "G") {
        gib = val                    # already GiB (ish)
    } else {
        # fallback: assume K if no unit
        gib = val / 1024 / 1024
    }

    if (NR == 1 || gib < min) min = gib
    if (NR == 1 || gib > max) max = gib
    sum += gib
    n++
}
END {
    mean = (n > 0) ? sum / n : 0
    printf "n=%d  min=%.2f GiB  mean=%.2f GiB  max=%.2f GiB\n",
            n, min, mean, max
}'

40166308,40166309,40166310,40166311,40166312,40166313,40166314,40166315,40166316,40166317,40166318,40166319,40166320,40166321,40166322,40166323,40166324,40166325,40166326,40166327,40166328
40166308.ba+                          batch  COMPLETED   00:01:58  00:47.316  15475320K 
40166309.ba+                          batch  COMPLETED   00:01:56  00:47.819  15476952K 
40166310.ba+                          batch  COMPLETED   00:01:54  00:43.897  15470564K 
40166311.ba+                          batch  COMPLETED   00:01:57  00:47.623  15472516K 
40166312.ba+                          batch  COMPLETED   00:01:57  00:47.697  15472272K 
40166313.ba+                          batch  COMPLETED   00:01:58  00:49.020  15473072K 
40166314.ba+                          batch  COMPLETED   00:01:56  00:46.472  15472940K 
40166315.ba+                          batch  COMPLETED   00:01:53  00:43.290  15474320K 
40166316.ba+                          batch  COMPLETED   00:00:46  00:41.182  15504004K 
40166317.b

In [50]:
cat ${FD_LOG}/run_motifdelta_scan_batch_pilot_001.txt

Loading FASTA sequences...
Loaded 5000 sequences
Load and check complete in 0.07 seconds

Loading motif matrices...
Loaded 879 motifs from /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/JASPAR2024_CORE_vertebrates_non-redundant.lods.pkl
Load and check complete in 0.01 seconds

Setting motif kernel...
Forward kernels shape: (879, 33, 4)
Reverse kernels shape: (879, 33, 4)
Set complete in 0.01 seconds

Running motif scanning...
Scan complete in 18.78 seconds
Estimated memory use (ref+obs+unobs): 7.269 GB

Saving results...
Saved results to /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifdelta_pilot/variant_closed_gof_bluestarr_flankL35R70_pilot_chunk001_scan.npz
Saved complete in 97.83 seconds

Done.


In [20]:
### set resource
NUM_CPU=5
NUM_MEM=20G

### loop each chunk and run motifdelta
for idx in $(seq 1 3); do

    ### prepare
    CHUNK=$(printf "%03d" "${idx}")
    PREFIX=variant_closed_gof_bluestarr_flankL35R70_pilot_chunk${CHUNK}
    echo "Running motif scan for ${PREFIX}..."

    ### execute
    FP_EXE=${FD_EXE}/run_motifdelta_01_scan.py
    FP_LOG=${FD_LOG}/run_motifdelta_scan_batch_pilot_${CHUNK}.txt
    echo '${FD_LOG}'/"run_motifdelta_scan_batch_pilot_${CHUNK}.txt"
    
    sbatch \
        -A majoroslab \
        -p igvf \
        --cpus-per-task ${NUM_CPU} \
        --mem    ${NUM_MEM} \
        --output ${FP_LOG}  \
        --chdir  ${FD_EXE}  \
        ${FP_APP} python ${FP_EXE} \
            --txt_fpath_fasta_ref  ${FD_BATCH}/${PREFIX}_ref.fa \
            --txt_fpath_fasta_obs  ${FD_BATCH}/${PREFIX}_obs.fa \
            --txt_fpath_fasta_ubs  ${FD_BATCH}/${PREFIX}_unobs.fa \
            --txt_fpath_motif      ${FP_MOTIF} \
            --txt_fpath_output     ${FD_DELTA}/${PREFIX}_scan.npz
done

Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk001...
${FD_LOG}/run_motifdelta_scan_batch_pilot_001.txt
Submitted batch job 40150536
Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk002...
${FD_LOG}/run_motifdelta_scan_batch_pilot_002.txt
Submitted batch job 40150538
Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk003...
${FD_LOG}/run_motifdelta_scan_batch_pilot_003.txt
Submitted batch job 40150539


In [21]:
cat ${FD_LOG}/run_motifdelta_scan_batch_pilot_001.txt

Loading FASTA sequences...
Loaded 5000 sequences

Load and check complete in 0.06 seconds

Loading motif matrices...
Loaded 879 motifs from /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/JASPAR2024_CORE_vertebrates_non-redundant.lods.pkl

Load and check complete in 0.01 seconds

Setting motif kernel...
Forward kernels shape: (879, 33, 4)
Reverse kernels shape: (879, 33, 4)
Set complete in 0.01 seconds

Running motif scanning...
Scan complete in 15.15 seconds

Estimated memory use (ref+obs+unobs): 7.269 GB
Saving results...
Saved results to /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifdelta_pilot/variant_closed_gof_bluestarr_flankL35R70_pilot_chunk001_scan.npz
Saved complete in 29.19 seconds

Done.


In [22]:
sacct -j 40150536 --format=JobID,MaxRSS,Elapsed

JobID            MaxRSS    Elapsed 
------------ ---------- ---------- 
40150536                  00:00:45 
40150536.ba+  15474556K   00:00:45 
40150536.ex+       256K   00:00:45 


In [23]:
ls -l /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifdelta_pilot/variant_closed_gof_bluestarr_flankL35R70_pilot_chunk001_scan.npz

-rw-r--r--. 1 kk319 majoroslab 7806022286 Nov 25 11:43 /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifdelta_pilot/variant_closed_gof_bluestarr_flankL35R70_pilot_chunk001_scan.npz


In [17]:
cat ${FD_LOG}/run_motifdelta_scan_batch_pilot_001.txt

Loading FASTA sequences...
Loaded 5000 sequences

Load and check complete in 0.06 seconds

Loading motif matrices...
Loaded 879 motifs from /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/JASPAR2024_CORE_vertebrates_non-redundant.lods.pkl

Load and check complete in 0.01 seconds

Setting motif kernel...
Forward kernels shape: (879, 33, 4)
Reverse kernels shape: (879, 33, 4)
Set complete in 0.01 seconds

Running motif scanning...
Scan complete in 15.88 seconds

Estimated memory use (ref+obs+unobs): 7.269 GB
Saving results...
Saved results to /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifdelta_pilot/variant_closed_gof_bluestarr_flankL35R70_pilot_chunk001_scan.npz
Saved complete in 355.87 seconds

Done.


In [18]:
sacct -j 40149712 --format=JobID,MaxRSS,Elapsed

JobID            MaxRSS    Elapsed 
------------ ---------- ---------- 
40149712                  00:06:13 
40149712.ba+  14622544K   00:06:13 
40149712.ex+          0   00:06:13 


In [19]:
ls -l /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifdelta_pilot/variant_closed_gof_bluestarr_flankL35R70_pilot_chunk001_scan.npz

-rw-r--r--. 1 kk319 majoroslab 6909817975 Nov 25 11:28 /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifdelta_pilot/variant_closed_gof_bluestarr_flankL35R70_pilot_chunk001_scan.npz


In [24]:
7 806 022 286
6909817975

bash: 7806022286: command not found


: 127

In [10]:
### set resource
NUM_CPU=5
NUM_MEM=20G

### loop each chunk and run motifdelta
for idx in $(seq 1 20); do

    ### prepare
    CHUNK=$(printf "%03d" "${idx}")
    PREFIX=variant_closed_gof_bluestarr_flankL35R70_pilot_chunk${CHUNK}
    echo "Running motif scan for ${PREFIX}..."

    ### execute
    FP_EXE=${FD_EXE}/run_motifdelta_01_scan.py
    FP_LOG=${FD_LOG}/run_motifdelta_scan_batch_pilot_${CHUNK}.txt
    echo '${FD_LOG}'/"run_motifdelta_scan_batch_pilot_${CHUNK}.txt"
    
    sbatch \
        -A majoroslab \
        -p igvf \
        --cpus-per-task ${NUM_CPU} \
        --mem    ${NUM_MEM} \
        --output ${FP_LOG}  \
        --chdir  ${FD_EXE}  \
        ${FP_APP} python ${FP_EXE} \
            --txt_fpath_fasta_ref  ${FD_BATCH}/${PREFIX}_ref.fa \
            --txt_fpath_fasta_obs  ${FD_BATCH}/${PREFIX}_obs.fa \
            --txt_fpath_fasta_ubs  ${FD_BATCH}/${PREFIX}_unobs.fa \
            --txt_fpath_motif      ${FP_MOTIF} \
            --txt_fpath_output     ${FD_DELTA}/${PREFIX}_scan.npz
done

Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk001...
${FD_LOG}/run_motifdelta_scan_batch_pilot_001.txt
Submitted batch job 40149247
Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk002...
${FD_LOG}/run_motifdelta_scan_batch_pilot_002.txt
Submitted batch job 40149248
Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk003...
${FD_LOG}/run_motifdelta_scan_batch_pilot_003.txt
Submitted batch job 40149249
Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk004...
${FD_LOG}/run_motifdelta_scan_batch_pilot_004.txt
Submitted batch job 40149253
Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk005...
${FD_LOG}/run_motifdelta_scan_batch_pilot_005.txt
Submitted batch job 40149254
Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk006...
${FD_LOG}/run_motifdelta_scan_batch_pilot_006.txt
Submitted batch job 40149255
Running motif scan for variant_clo

## Review

In [13]:
cat ${FD_LOG}/run_motifdelta_scan_batch_pilot_001.txt

Loaded 5000 sequences

Loaded 879 motifs from /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/JASPAR2024_CORE_vertebrates_non-redundant.lods.pkl

Forward kernels shape: (879, 33, 4)
Reverse kernels shape: (879, 33, 4)
Running motif scanning...
Scan complete in 18.14 seconds

Estimated memory use (ref+obs+unobs): 7.269 GB
Saved results to /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifdelta_pilot/variant_closed_gof_bluestarr_flankL35R70_pilot_chunk001_scan.npz
Done.


In [15]:
sacct -j 40149247 --format=JobID,MaxRSS,Elapsed

JobID            MaxRSS    Elapsed 
------------ ---------- ---------- 
40149247                  00:06:18 
40149247.ba+  14626648K   00:06:18 
40149247.ex+          0   00:06:18 


In [14]:
sacct -j 40149117 --format=JobID,MaxRSS,Elapsed

JobID            MaxRSS    Elapsed 
------------ ---------- ---------- 
40149117                  00:00:35 
40149117.ba+  12581728K   00:00:35 
40149117.ex+       256K   00:00:35 


In [ ]:
Loaded 5000 sequences

Loaded 879 motifs from /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/JASPAR2024_CORE_vertebrates_non-redundant.lods.pkl

Forward kernels shape: (879, 33, 4)
Reverse kernels shape: (879, 33, 4)
Running motif scanning...
Scan complete in 18.14 seconds

Estimated memory use (ref+obs+unobs): 7.269 GB
Saved results to /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifdelta_pilot/variant_closed_gof_bluestarr_flankL35R70_pilot_chunk001_scan.npz
Done.